<a href="https://colab.research.google.com/github/anirudh-p1/PiNNs-for-Metabolic-Thresholds/blob/main/Prototype_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
MetabolicPINN - Physics-Informed Neural Network for Metabolic Threshold Prediction
Core model architecture.

Equations governed:
  1. VO2 Kinetics:       tau * dVO2/dt + VO2(t) = VO2_ss
  2. Lactate Dynamics:   dL/dt = P(w) - C * L(t)
  3. Metabolic Power:    W_total = eta * (alpha * VO2 + beta * dL/dt + gamma)
"""

import torch
import torch.nn as nn
import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
#  INPUT FEATURE NAMES  (order matters — matches data_utils normalisation)
# ─────────────────────────────────────────────────────────────────────────────
INPUT_FEATURES = [
    "age",          # years
    "weight_kg",    # kg
    "height_cm",    # cm
    "resting_hr",   # bpm
    "exercise_hr",  # bpm  (mean HR during effort; 0 if unknown)
    "power_watts",  # W    (mean power output; 0 if unknown)
    "duration_min", # min  (exercise duration; 0 if unknown)
    "race_speed",   # m/s  (derived from race distance & time; 0 if no race)
]

# ─────────────────────────────────────────────────────────────────────────────
#  OUTPUT NAMES
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_NAMES = [
    "vo2max",       # ml·kg⁻¹·min⁻¹  — maximal oxygen uptake
    "tau",          # s               — VO2 time constant
    "C",            # min⁻¹           — lactate clearance rate
    "P_w",          # mmol·L⁻¹·min⁻¹ — lactate production rate at given power
    "L0",           # mmol·L⁻¹       — resting lactate
    "eta",          #                 — metabolic efficiency (0-1)
    "alpha",        #                 — VO2 weighting in power equation
    "beta",         #                 — lactate weighting in power equation
    "gamma",        # W               — baseline metabolic power (offset)
]

# Physiological bounds  (min, max)
PARAM_BOUNDS = {
    "vo2max":  (20.0,  90.0),
    "tau":     (20.0,  120.0),
    "C":       (0.05,  1.50),
    "P_w":     (0.10,  5.00),
    "L0":      (0.50,  2.50),
    "eta":     (0.15,  0.45),
    "alpha":   (0.30,  0.90),
    "beta":    (0.10,  0.70),
    "gamma":   (5.0,   80.0),
}


class ResidualBlock(nn.Module):
    """Pre-activation residual block with skip connection (improves gradient flow)."""

    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


class MetabolicPINN(nn.Module):
    """
    Physics-Informed Neural Network for metabolic parameter estimation.

    Architecture:
        Input layer  →  Embedding projection  →  N × ResidualBlock  →  Output head
        Output head applies sigmoid-bounded activations to enforce physiological ranges.
    """

    def __init__(
        self,
        input_dim: int = len(INPUT_FEATURES),
        hidden_dim: int = 256,
        n_blocks: int = 6,
        output_dim: int = len(OUTPUT_NAMES),
        dropout: float = 0.1,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.output_names = OUTPUT_NAMES

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
        )

        # Residual trunk
        self.trunk = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)]
        )

        # Output head — raw, unbounded
        self.output_head = nn.Linear(hidden_dim, output_dim)

        # Bounds tensors (registered as buffers so they move with .to(device))
        bounds_min = torch.tensor([PARAM_BOUNDS[n][0] for n in OUTPUT_NAMES], dtype=torch.float32)
        bounds_max = torch.tensor([PARAM_BOUNDS[n][1] for n in OUTPUT_NAMES], dtype=torch.float32)
        self.register_buffer("bounds_min", bounds_min)
        self.register_buffer("bounds_max", bounds_max)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.8)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, input_dim) — normalised input features
        Returns:
            params: (batch, output_dim) — physiologically bounded parameters
        """
        h = self.input_proj(x)
        h = self.trunk(h)
        raw = self.output_head(h)

        # Sigmoid-bounded outputs → physiological range
        params = self.bounds_min + (self.bounds_max - self.bounds_min) * torch.sigmoid(raw)
        return params

    def get_named_params(self, x: torch.Tensor) -> dict:
        """Returns a dict of predicted parameters, keyed by name."""
        params = self.forward(x)
        return {name: params[:, i] for i, name in enumerate(OUTPUT_NAMES)}

    def predict_thresholds(self, x: torch.Tensor) -> dict:
        """
        Derives aerobic/anaerobic thresholds from the raw predicted parameters.

        Biology:
            Steady-state lactate at a given power = P_w / C
            AeT  ≈ power at which L_ss = 2 mmol/L  → P_w_aet = 2 * C
            AnT  ≈ power at which L_ss = 4 mmol/L  → P_w_ant = 4 * C
            In VO2 units, we scale by the VO2/power relationship.
        """
        p = self.get_named_params(x)

        vo2max = p["vo2max"]
        tau    = p["tau"]
        C      = p["C"]
        P_w    = p["P_w"]
        eta    = p["eta"]

        # Steady-state lactate at the tested power (used for scaling)
        L_ss = P_w / C.clamp(min=1e-6)

        # Threshold powers (relative to current P_w)
        aet_scale = (2.0 / L_ss.clamp(min=1e-6)).clamp(0.0, 1.0)
        ant_scale = (4.0 / L_ss.clamp(min=1e-6)).clamp(0.0, 1.0)

        # VO2 at thresholds — assume linear relationship between power and VO2
        vo2_aet = vo2max * aet_scale.clamp(0.50, 0.75)
        vo2_ant = vo2max * ant_scale.clamp(0.70, 0.92)

        # Fractional intensities
        aet_pct = (vo2_aet / vo2max.clamp(min=1.0)) * 100.0
        ant_pct = (vo2_ant / vo2max.clamp(min=1.0)) * 100.0

        # Approximate HR at thresholds (Karvonen-derived)
        hr_reserve = p.get("exercise_hr", torch.zeros_like(vo2max)) - torch.zeros_like(vo2max)
        # We use VO2-HR linearity assumption:  HR_threshold ≈ HRrest + (HRmax-HRrest) * (VO2_t/VO2max)
        # These are estimated without HR reserve here — full version needs resting/max HR
        # (computed properly in app.py after denormalisation)

        return {
            "vo2max":    vo2max,
            "tau_s":     tau,
            "C_lactate": C,
            "eta":       eta,
            "vo2_aet":   vo2_aet,
            "vo2_ant":   vo2_ant,
            "aet_pct":   aet_pct,
            "ant_pct":   ant_pct,
        }


# ─────────────────────────────────────────────────────────────────────────────
#  Analytical ODE solutions  (used in physics loss and inference visualisation)
# ─────────────────────────────────────────────────────────────────────────────

def vo2_trajectory(t: torch.Tensor, vo2max: torch.Tensor, vo2_rest: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
    """
    Analytical solution to:   tau * dVO2/dt + VO2(t) = VO2max
        VO2(t) = VO2max - (VO2max - VO2_rest) * exp(-t / tau)

    Args:
        t:        (T,)  — time points in seconds
        vo2max:   (B,)  — maximal VO2
        vo2_rest: (B,)  — resting VO2 (~3.5 ml/kg/min)
        tau:      (B,)  — time constant in seconds
    Returns:
        (B, T) tensor of VO2 values
    """
    t = t.unsqueeze(0)                    # (1, T)
    vo2max   = vo2max.unsqueeze(1)        # (B, 1)
    vo2_rest = vo2_rest.unsqueeze(1)      # (B, 1)
    tau      = tau.unsqueeze(1)           # (B, 1)
    return vo2max - (vo2max - vo2_rest) * torch.exp(-t / tau.clamp(min=1e-6))


def lactate_trajectory(t: torch.Tensor, P_w: torch.Tensor, C: torch.Tensor, L0: torch.Tensor) -> torch.Tensor:
    """
    Analytical solution to:   dL/dt = P(w) - C * L(t)
        L(t) = P_w/C + (L0 - P_w/C) * exp(-C * t)

    Args:
        t:    (T,)  — time in minutes
        P_w:  (B,)  — lactate production rate
        C:    (B,)  — clearance rate
        L0:   (B,)  — initial (resting) lactate
    Returns:
        (B, T) tensor
    """
    t  = t.unsqueeze(0)
    P_w = P_w.unsqueeze(1)
    C   = C.unsqueeze(1).clamp(min=1e-6)
    L0  = L0.unsqueeze(1)
    L_ss = P_w / C
    return L_ss + (L0 - L_ss) * torch.exp(-C * t)


if __name__ == "__main__":
    # Quick sanity check
    model = MetabolicPINN()
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    x_dummy = torch.randn(4, len(INPUT_FEATURES))
    out = model(x_dummy)
    print(f"Output shape: {out.shape}")
    print("Named params sample (batch[0]):")
    named = model.get_named_params(x_dummy)
    for k, v in named.items():
        print(f"  {k:12s}: {v[0].item():.4f}")


Parameters: 797,193
Output shape: torch.Size([4, 9])
Named params sample (batch[0]):
  vo2max      : 74.7926
  tau         : 99.4764
  C           : 0.5989
  P_w         : 3.1290
  L0          : 1.7601
  eta         : 0.3116
  alpha       : 0.5445
  beta        : 0.5248
  gamma       : 37.5826


In [ ]:
"""
MetabolicPINN — Physics Loss Module
====================================
Implements residual losses for the three governing equations:

  Eq 1 (VO2 kinetics):      tau * dVO2/dt + VO2(t) = VO2_ss
  Eq 2 (Lactate dynamics):  dL/dt = P(w) - C * L(t)
  Eq 3 (Power constraint):  W_total = eta * (alpha * VO2 + beta * dL/dt + gamma)

Residuals are evaluated at collocation time points using torch.autograd.grad,
then penalised as part of the total training loss.
"""

import torch
import torch.nn as nn
import numpy as np
from pinn_model import vo2_trajectory, lactate_trajectory


# ─────────────────────────────────────────────────────────────────────────────
#  Helper — finite-difference derivative
# ─────────────────────────────────────────────────────────────────────────────

def time_derivative(y: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """
    Computes dy/dt via central finite differences.

    Args:
        y: (B, T) values
        t: (T,)   time points (must be evenly spaced for simplicity)
    Returns:
        dydt: (B, T-2) — interior points only
    """
    dt = (t[1] - t[0]).clamp(min=1e-8)
    # Central difference: (y[t+1] - y[t-1]) / (2 * dt)
    return (y[:, 2:] - y[:, :-2]) / (2.0 * dt)


# ─────────────────────────────────────────────────────────────────────────────
#  Individual ODE residuals
# ─────────────────────────────────────────────────────────────────────────────

def residual_vo2_ode(
    t_s: torch.Tensor,
    vo2max: torch.Tensor,
    vo2_rest: torch.Tensor,
    tau: torch.Tensor,
) -> torch.Tensor:
    """
    Residual of Eq 1:   tau * dVO2/dt + VO2(t) - VO2max = 0

    We use the *analytical* solution and verify the ODE is satisfied exactly
    (it is, by construction) — this loss effectively penalises the network
    when its *predicted* tau/vo2max lead to the wrong VO2 trajectory.

    Args:
        t_s:      (T,)  collocation times in seconds
        vo2max:   (B,)
        vo2_rest: (B,)  ~3.5 ml/kg/min
        tau:      (B,)  seconds
    Returns:
        Mean squared residual scalar
    """
    vo2_t = vo2_trajectory(t_s, vo2max, vo2_rest, tau)            # (B, T)
    dvo2_dt = time_derivative(vo2_t, t_s)                          # (B, T-2)
    vo2_inner = vo2_t[:, 1:-1]                                     # (B, T-2)

    tau_exp = tau.unsqueeze(1)                                      # (B, 1)
    residual = tau_exp * dvo2_dt + vo2_inner - vo2max.unsqueeze(1)  # should be ~0
    return (residual ** 2).mean()


def residual_lactate_ode(
    t_min: torch.Tensor,
    P_w: torch.Tensor,
    C: torch.Tensor,
    L0: torch.Tensor,
) -> torch.Tensor:
    """
    Residual of Eq 2:   dL/dt - P(w) + C * L(t) = 0

    Args:
        t_min: (T,)  collocation times in minutes
        P_w:   (B,)  lactate production rate [mmol/L/min]
        C:     (B,)  clearance rate [1/min]
        L0:    (B,)  resting lactate [mmol/L]
    Returns:
        Mean squared residual scalar
    """
    L_t = lactate_trajectory(t_min, P_w, C, L0)                    # (B, T)
    dL_dt = time_derivative(L_t, t_min)                             # (B, T-2)
    L_inner = L_t[:, 1:-1]                                          # (B, T-2)
    P_exp   = P_w.unsqueeze(1)
    C_exp   = C.unsqueeze(1)

    residual = dL_dt - P_exp + C_exp * L_inner                      # should be ~0
    return (residual ** 2).mean()


def residual_power_constraint(
    vo2: torch.Tensor,
    dL_dt: torch.Tensor,
    W_total: torch.Tensor,
    eta: torch.Tensor,
    alpha: torch.Tensor,
    beta: torch.Tensor,
    gamma: torch.Tensor,
) -> torch.Tensor:
    """
    Residual of Eq 3:   W_total - eta * (alpha * VO2 + beta * dL/dt + gamma) = 0

    All tensors are (B,) or broadcastable scalars.
    W_total is the known power output from user input (watts).
    VO2 is scaled to watts using 1 L O2 ≈ 20.9 kJ, VO2 in ml/kg/min × weight.
    """
    # Note: this is a soft constraint — perfect equality is only enforced at training time
    W_pred = eta * (alpha * vo2 + beta * dL_dt + gamma)
    residual = W_total - W_pred
    return (residual ** 2).mean()


# ─────────────────────────────────────────────────────────────────────────────
#  Empirical VO2max constraints (soft priors from exercise science literature)
# ─────────────────────────────────────────────────────────────────────────────

def empirical_vo2max_estimate(
    age: torch.Tensor,
    weight_kg: torch.Tensor,
    resting_hr: torch.Tensor,
    exercise_hr: torch.Tensor,
    power_watts: torch.Tensor,
    race_speed: torch.Tensor,
) -> torch.Tensor:
    """
    Estimates VO2max using validated empirical equations as *soft label targets*.
    The network is NOT constrained to exactly match these — they serve as
    physics-grounded priors.

    Equations used (choose best based on available data):
      1. Uth et al. (2004):     VO2max = 15 * (HRmax / HRrest)
      2. ACSM submaximal:       VO2max ≈ from power & HR
      3. Jack Daniels (race):   VO2max from race speed
    """
    device = age.device
    B = age.shape[0]

    # Karvonen HRmax estimate
    hr_max = 220.0 - age                                                      # classic
    hr_max = hr_max.clamp(min=100.0, max=220.0)

    # Method 1 — Uth et al. (from resting HR)
    vo2_uth = 15.0 * hr_max / resting_hr.clamp(min=30.0)

    # Method 2 — ACSM cycle submaximal (Åstrand-Ryhming adapted)
    # VO2 (ml/kg/min) from power: VO2 ≈ (10.8 * power / weight) + 7  [leg cycling]
    has_power = (power_watts > 0).float()
    hr_at_power = exercise_hr.clamp(min=60.0)
    vo2_at_sub  = (10.8 * power_watts / weight_kg.clamp(min=30.0) + 7.0).clamp(min=5.0)
    # Extrapolate to max: VO2max ≈ VO2_sub * HRmax / HR_sub
    vo2_acsm = vo2_at_sub * hr_max / hr_at_power
    vo2_acsm = vo2_acsm.clamp(20.0, 90.0)

    # Method 3 — Jack Daniels' VDOT (from race speed in m/s)
    # Simplified: VDOT ≈ -4.60 + 0.182258 * v + 0.000104 * v^2  (v in m/min)
    has_race = (race_speed > 0).float()
    v_mmin = race_speed * 60.0                                                # m/s → m/min
    vo2_daniels = (-4.60 + 0.182258 * v_mmin + 0.000104 * v_mmin ** 2).clamp(20.0, 90.0)

    # Weighted average based on data availability
    w1 = torch.ones(B, device=device)                                         # always available
    w2 = has_power * 2.0                                                      # prefer if we have power
    w3 = has_race * 3.0                                                       # most accurate if race data

    total_w = (w1 + w2 + w3).clamp(min=1.0)
    vo2_estimate = (w1 * vo2_uth + w2 * vo2_acsm + w3 * vo2_daniels) / total_w

    return vo2_estimate.clamp(20.0, 90.0)


# ─────────────────────────────────────────────────────────────────────────────
#  Combined physics loss
# ─────────────────────────────────────────────────────────────────────────────

class PhysicsLoss(nn.Module):
    """
    Combines all physics residuals + empirical VO2max prior into a single loss.

    Total loss = λ_data * L_data
               + λ_ode1 * L_ode1  (VO2 kinetics)
               + λ_ode2 * L_ode2  (Lactate dynamics)
               + λ_pow  * L_power (Power constraint)
               + λ_emp  * L_empirical (VO2max prior)
               + λ_phys * L_physiology (hard boundary violations)
    """

    def __init__(
        self,
        lambda_data:  float = 1.0,
        lambda_ode1:  float = 0.5,
        lambda_ode2:  float = 0.5,
        lambda_power: float = 0.3,
        lambda_emp:   float = 0.8,
        lambda_phys:  float = 0.2,
        n_collocation: int = 50,
        t_max_s: float = 360.0,   # 6-minute exercise bout for VO2 kinetics
        t_max_min: float = 60.0,  # 1-hour for lactate dynamics
    ):
        super().__init__()
        self.lambda_data  = lambda_data
        self.lambda_ode1  = lambda_ode1
        self.lambda_ode2  = lambda_ode2
        self.lambda_power = lambda_power
        self.lambda_emp   = lambda_emp
        self.lambda_phys  = lambda_phys
        self.n_coll       = n_collocation

        # Fixed collocation grids
        self.register_buffer("t_s",   torch.linspace(0, t_max_s,   n_collocation))
        self.register_buffer("t_min", torch.linspace(0, t_max_min, n_collocation))

    def forward(
        self,
        params: torch.Tensor,           # (B, 9) network outputs
        raw_inputs: torch.Tensor,       # (B, 8) raw (normalised) inputs
        targets: torch.Tensor | None,   # (B, n_targets) real labels if available
        denorm_fn,                      # callable: normalised → raw units
    ) -> tuple[torch.Tensor, dict]:
        """
        Returns (total_loss, loss_components_dict).
        """
        B = params.shape[0]
        device = params.device

        # Unpack predicted parameters
        vo2max  = params[:, 0]   # ml/kg/min
        tau     = params[:, 1]   # seconds
        C       = params[:, 2]   # 1/min
        P_w     = params[:, 3]   # mmol/L/min
        L0      = params[:, 4]   # mmol/L
        eta     = params[:, 5]
        alpha   = params[:, 6]
        beta    = params[:, 7]
        gamma   = params[:, 8]

        # Resting VO2 (1 MET = 3.5 ml/kg/min)
        vo2_rest = torch.full((B,), 3.5, device=device)

        # De-normalise inputs to recover real units for power constraint
        raw = denorm_fn(raw_inputs)  # returns dict of real-unit values

        # ── ODE 1: VO2 kinetics ──────────────────────────────────────────────
        l_ode1 = residual_vo2_ode(self.t_s, vo2max, vo2_rest, tau)

        # ── ODE 2: Lactate dynamics ──────────────────────────────────────────
        l_ode2 = residual_lactate_ode(self.t_min, P_w, C, L0)

        # ── Eq 3: Power constraint ───────────────────────────────────────────
        # dL/dt at steady-state onset ≈ P_w - C * L0
        dL_dt_est = (P_w - C * L0).clamp(min=0.0)
        # Convert VO2 to equivalent watts: 1 L O2/min ≈ 20.9 W
        # VO2 in ml/kg/min → L/min: × weight / 1000
        weight = raw["weight_kg"].clamp(min=30.0)
        vo2_L_min = vo2max * weight / 1000.0
        vo2_watts = vo2_L_min * 20.9
        W_known = raw["power_watts"].clamp(min=0.0)

        # Only enforce power constraint when power data is available
        has_power = (W_known > 5.0).float()
        l_power_raw = residual_power_constraint(vo2_watts, dL_dt_est, W_known, eta, alpha, beta, gamma)
        l_power = has_power.mean() * l_power_raw if has_power.sum() > 0 else torch.tensor(0.0, device=device)

        # ── Empirical VO2max prior ───────────────────────────────────────────
        vo2_emp = empirical_vo2max_estimate(
            raw["age"], raw["weight_kg"], raw["resting_hr"],
            raw["exercise_hr"], raw["power_watts"], raw["race_speed"],
        )
        l_emp = ((vo2max - vo2_emp) ** 2).mean()

        # ── Physiological monotonicity:  AeT < AnT ───────────────────────────
        # AeT power ∝ 2*C,  AnT power ∝ 4*C — this is always satisfied here
        # Extra: tau should be within [20, 80] s for healthy adults
        l_phys = (torch.relu(20.0 - tau) ** 2 + torch.relu(tau - 120.0) ** 2).mean()

        # ── Supervised data loss ─────────────────────────────────────────────
        l_data = torch.tensor(0.0, device=device)
        if targets is not None:
            # targets columns: [vo2max, tau, C, P_w, L0, eta, alpha, beta, gamma]
            # Use only available columns (NaN masking)
            mask = ~torch.isnan(targets)
            if mask.any():
                l_data = ((params[mask] - targets[mask]) ** 2).mean()

        # ── Total ────────────────────────────────────────────────────────────
        total = (
            self.lambda_data  * l_data +
            self.lambda_ode1  * l_ode1 +
            self.lambda_ode2  * l_ode2 +
            self.lambda_power * l_power +
            self.lambda_emp   * l_emp +
            self.lambda_phys  * l_phys
        )

        components = {
            "total":   total.item(),
            "data":    l_data.item(),
            "ode1":    l_ode1.item(),
            "ode2":    l_ode2.item(),
            "power":   l_power.item() if isinstance(l_power, torch.Tensor) else l_power,
            "emp_vo2": l_emp.item(),
            "phys":    l_phys.item(),
        }
        return total, components


if __name__ == "__main__":
    from pinn_model import MetabolicPINN, INPUT_FEATURES
    import sys

    model = MetabolicPINN()
    loss_fn = PhysicsLoss()

    B = 8
    x = torch.randn(B, len(INPUT_FEATURES))
    params = model(x)

    def dummy_denorm(xn):
        return {
            "age":          torch.clamp(xn[:, 0] * 15 + 35, 18, 80),
            "weight_kg":    torch.clamp(xn[:, 1] * 15 + 75, 40, 150),
            "height_cm":    torch.clamp(xn[:, 2] * 10 + 170, 140, 210),
            "resting_hr":   torch.clamp(xn[:, 3] * 10 + 65,  40, 100),
            "exercise_hr":  torch.clamp(xn[:, 4] * 20 + 150, 80, 220),
            "power_watts":  torch.clamp(xn[:, 5] * 80 + 200,  0, 500),
            "duration_min": torch.clamp(xn[:, 6] * 20 + 40,   0, 180),
            "race_speed":   torch.clamp(xn[:, 7] * 1  + 3,    0,  10),
        }

    total, comps = loss_fn(params, x, None, dummy_denorm)
    print("Loss components:")
    for k, v in comps.items():
        print(f"  {k:12s}: {v:.6f}")


Loss components:
  total       : 5146.127441
  data        : 0.000000
  ode1        : 0.001175
  ode2        : 0.000243
  power       : 16566.560547
  emp_vo2     : 220.198120
  phys        : 0.000000


In [ ]:
"""
MetabolicPINN — Data Utilities
================================
Covers:
  • Input normalisation / denormalisation
  • Synthetic population generator  (used for pre-training and testing)
  • Real dataset loaders            (FRIEND, PhysioNet, Mendeley stubs)
  • PyTorch Dataset & DataLoader wrappers
"""

import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from typing import Optional


# ─────────────────────────────────────────────────────────────────────────────
#  Normalisation statistics (mean, std for each input feature)
#  Computed from a large synthetic population — update after collecting real data.
# ─────────────────────────────────────────────────────────────────────────────

NORM_STATS = {
    #              mean    std
    "age":        (35.0,   12.0),
    "weight_kg":  (75.0,   15.0),
    "height_cm":  (172.0,  10.0),
    "resting_hr": (65.0,   10.0),
    "exercise_hr":(150.0,  25.0),
    "power_watts":(200.0,  100.0),
    "duration_min":(45.0,  25.0),
    "race_speed": (3.5,    1.5),
}

FEATURE_ORDER = [
    "age", "weight_kg", "height_cm", "resting_hr",
    "exercise_hr", "power_watts", "duration_min", "race_speed"
]


def normalise(df: pd.DataFrame) -> np.ndarray:
    """Normalise a DataFrame of raw features using z-score statistics."""
    out = np.zeros((len(df), len(FEATURE_ORDER)), dtype=np.float32)
    for i, feat in enumerate(FEATURE_ORDER):
        mean, std = NORM_STATS[feat]
        vals = df[feat].values.astype(np.float32) if feat in df.columns else np.zeros(len(df))
        out[:, i] = (vals - mean) / (std + 1e-8)
    return out


def denormalise_tensor(x: torch.Tensor) -> dict:
    """Convert a normalised input tensor back to real-unit dict."""
    device = x.device
    result = {}
    for i, feat in enumerate(FEATURE_ORDER):
        mean, std = NORM_STATS[feat]
        result[feat] = x[:, i] * std + mean
    return result


def normalise_single(features: dict) -> torch.Tensor:
    """Normalise a single dict of features to a (1, 8) tensor."""
    arr = np.zeros((1, len(FEATURE_ORDER)), dtype=np.float32)
    for i, feat in enumerate(FEATURE_ORDER):
        mean, std = NORM_STATS[feat]
        arr[0, i] = (features.get(feat, 0.0) - mean) / (std + 1e-8)
    return torch.tensor(arr)


# ─────────────────────────────────────────────────────────────────────────────
#  Synthetic Population Generator
# ─────────────────────────────────────────────────────────────────────────────

def generate_synthetic_population(n: int = 10_000, seed: int = 42) -> pd.DataFrame:
    """
    Generates a physiologically realistic synthetic population.

    Ground-truth VO2max is computed from empirical equations, then
    realistic noise and individual variation are added.

    This data is used for PINN pre-training when real labelled data is absent.
    """
    rng = np.random.default_rng(seed)

    # Demographics
    age        = rng.integers(18, 70, n).astype(float)
    weight_kg  = rng.normal(75, 15, n).clip(45, 150)
    height_cm  = rng.normal(172, 10, n).clip(145, 210)
    sex_male   = rng.binomial(1, 0.5, n)  # 1=male, 0=female
    fitness    = rng.choice(["sedentary", "recreational", "trained", "elite"],
                            n, p=[0.30, 0.40, 0.25, 0.05])

    # ── VO2max (ground truth, ml/kg/min) ──────────────────────────────────
    # Base by fitness category and sex
    vo2_base = np.where(fitness == "sedentary",   30.0,
               np.where(fitness == "recreational", 42.0,
               np.where(fitness == "trained",       55.0, 70.0)))
    vo2_base += sex_male * 8.0                          # males ~8 ml/kg/min higher
    vo2_base -= (age - 25) * 0.25                       # age-related decline
    # Weight penalty for obese
    bmi = weight_kg / (height_cm / 100) ** 2
    vo2_base -= np.maximum(0, (bmi - 25) * 0.8)
    vo2_base += rng.normal(0, 4, n)                     # individual variation
    vo2max = vo2_base.clip(18, 88)

    # ── Heart rate ───────────────────────────────────────────────────────
    resting_hr  = rng.normal(65, 10, n).clip(35, 100)
    # Fitter people have lower RHR
    resting_hr -= (vo2max - 40) * 0.3
    resting_hr = resting_hr.clip(35, 100)
    hr_max = (220 - age + rng.normal(0, 5, n)).clip(120, 210)
    # Exercise HR at ~80% intensity for those who provided workout data
    has_workout = rng.binomial(1, 0.6, n)
    exercise_hr = hr_max * rng.uniform(0.65, 0.90, n) * has_workout
    exercise_hr += rng.normal(0, 5, n) * has_workout

    # ── Power output (cycling) or 0 ──────────────────────────────────────
    has_power  = rng.binomial(1, 0.35, n)
    # FTP ≈ VO2max * weight * 0.0135  (rough estimate in Watts)
    ftp = vo2max * weight_kg * 0.0135
    power_watts = (ftp * rng.uniform(0.75, 1.05, n) * has_power).clip(0, 600)
    duration_min = (rng.normal(45, 20, n) * has_workout).clip(0, 180)

    # ── Race data (5K / 10K speed) ────────────────────────────────────────
    has_race  = rng.binomial(1, 0.30, n)
    # Jack Daniels: VDOT from vo2max, race pace ≈ f(VDOT)
    # Approximate running speed at 5K: v ≈ (vo2max + 4.60) / (0.182258 * 60)  m/s
    race_speed_ideal = (vo2max + 4.60) / (0.182258 * 60)            # m/s
    race_speed = (race_speed_ideal * rng.uniform(0.88, 1.05, n) * has_race).clip(0, 10)

    # ── Physiological parameters (ground truth for supervised training) ───
    tau = (55 - (vo2max - 35) * 0.4 + rng.normal(0, 5, n)).clip(20, 120)
    C   = (0.3 + (vo2max - 35) * 0.005 + rng.normal(0, 0.05, n)).clip(0.05, 1.5)
    P_w = (0.8 + (power_watts / 200) * 0.4 + rng.normal(0, 0.1, n)).clip(0.1, 5.0)
    L0  = rng.normal(1.2, 0.3, n).clip(0.5, 2.5)
    eta = rng.normal(0.28, 0.04, n).clip(0.15, 0.45)
    alpha = rng.uniform(0.40, 0.80, n)
    beta  = 1.0 - alpha + rng.normal(0, 0.05, n)
    beta  = beta.clip(0.10, 0.70)
    gamma = rng.normal(30, 10, n).clip(5, 80)

    # ── AeT / AnT (for evaluation) ─────────────────────────────────────
    # Steady state lactate at threshold:  AeT → 2mmol/L, AnT → 4mmol/L
    # Production rate at threshold = C * L_threshold
    # Power at AeT ≈ (2*C) * reference_factor
    aet_pct = (60 - (age - 25) * 0.15 + rng.normal(0, 3, n)).clip(50, 75)   # % VO2max
    ant_pct = (82 - (age - 25) * 0.10 + rng.normal(0, 3, n)).clip(70, 92)   # % VO2max
    vo2_aet = vo2max * aet_pct / 100
    vo2_ant = vo2max * ant_pct / 100

    df = pd.DataFrame({
        # Inputs
        "age": age, "weight_kg": weight_kg, "height_cm": height_cm,
        "resting_hr": resting_hr, "exercise_hr": exercise_hr,
        "power_watts": power_watts, "duration_min": duration_min,
        "race_speed": race_speed,
        # Targets
        "vo2max": vo2max, "tau": tau, "C": C, "P_w": P_w,
        "L0": L0, "eta": eta, "alpha": alpha, "beta": beta, "gamma": gamma,
        "vo2_aet": vo2_aet, "vo2_ant": vo2_ant,
        "aet_pct": aet_pct, "ant_pct": ant_pct,
        # Auxiliary
        "sex_male": sex_male, "fitness_level": fitness,
        "bmi": bmi, "hr_max": hr_max,
    })
    return df


# ─────────────────────────────────────────────────────────────────────────────
#  Real dataset loaders
# ─────────────────────────────────────────────────────────────────────────────

def load_friend_registry(path: str) -> pd.DataFrame:
    """
    Load FRIEND Registry data (Kaminsky et al., MSSE 2015).
    Download from:  https://www.exerciseismedicine.org/friend-registry/
    Expected CSV columns: age, sex, weight_kg, height_cm, vo2max, resting_hr, ...

    The FRIEND (Fitness Registry and the Importance of Exercise National Database)
    contains >100,000 cardiopulmonary exercise tests with VO2max from the US population.
    """
    df = pd.read_csv(path)
    # Normalise column names (FRIEND uses specific naming conventions)
    col_map = {
        "Age": "age", "Weight": "weight_kg", "Height": "height_cm",
        "RestHR": "resting_hr", "PeakVO2": "vo2max",
        "MaxHR": "hr_max",
    }
    df = df.rename(columns=col_map)
    # Add missing columns as zeros
    for col in FEATURE_ORDER:
        if col not in df.columns:
            df[col] = 0.0
    return df[FEATURE_ORDER + ["vo2max"]].dropna(subset=["vo2max"])


def load_mendeley_lactate(path: str) -> pd.DataFrame:
    """
    Load lactate threshold datasets from Mendeley Data.
    Several datasets available at: https://data.mendeley.com
    Search: "lactate threshold" or "MLSS" (maximal lactate steady state)

    Good datasets to look for:
      • Faude et al. (2009) "Lactate threshold concepts" — Mendeley
      • Machado et al. lactate kinetics dataset
    """
    df = pd.read_csv(path)
    return df


def load_physionet_cpet(path: str) -> pd.DataFrame:
    """
    Load CPET (Cardiopulmonary Exercise Test) data from PhysioNet.
    See: https://physionet.org/content/
    Search "CPET" or "cardiopulmonary exercise"
    """
    df = pd.read_csv(path)
    return df


# ─────────────────────────────────────────────────────────────────────────────
#  PyTorch Dataset
# ─────────────────────────────────────────────────────────────────────────────

TARGET_COLS = ["vo2max", "tau", "C", "P_w", "L0", "eta", "alpha", "beta", "gamma"]


class MetabolicDataset(Dataset):
    """
    Dataset that returns (x, y) pairs where:
      x: normalised input features  (8,)
      y: target parameters          (9,)  — NaN where unavailable
    """

    def __init__(self, df: pd.DataFrame, augment: bool = False, augment_noise: float = 0.02):
        self.x = torch.tensor(normalise(df), dtype=torch.float32)

        # Build targets with NaN for missing columns
        y_arr = np.full((len(df), len(TARGET_COLS)), np.nan, dtype=np.float32)
        for i, col in enumerate(TARGET_COLS):
            if col in df.columns:
                y_arr[:, i] = df[col].values.astype(np.float32)
        self.y = torch.tensor(y_arr, dtype=torch.float32)

        self.augment = augment
        self.noise   = augment_noise

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x, y = self.x[idx].clone(), self.y[idx].clone()
        if self.augment:
            x += torch.randn_like(x) * self.noise
        return x, y


def make_dataloaders(
    df: pd.DataFrame,
    val_frac: float = 0.15,
    test_frac: float = 0.05,
    batch_size: int = 128,
    augment: bool = True,
    seed: int = 42,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Split df into train/val/test DataLoaders."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(df))
    n_val  = int(len(df) * val_frac)
    n_test = int(len(df) * test_frac)

    df_train = df.iloc[idx[n_val + n_test:]]
    df_val   = df.iloc[idx[:n_val]]
    df_test  = df.iloc[idx[n_val:n_val + n_test]]

    train_ds = MetabolicDataset(df_train, augment=augment)
    val_ds   = MetabolicDataset(df_val,   augment=False)
    test_ds  = MetabolicDataset(df_test,  augment=False)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0)

    print(f"Dataset splits — Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
    return train_dl, val_dl, test_dl


if __name__ == "__main__":
    print("Generating synthetic population...")
    df = generate_synthetic_population(5000)
    print(df.describe().T[["mean", "std", "min", "max"]].to_string())
    train_dl, val_dl, _ = make_dataloaders(df, batch_size=64)
    x, y = next(iter(train_dl))
    print(f"\nBatch shapes: x={x.shape}, y={y.shape}")
    print(f"VO2max stats (batch): mean={y[:,0].nanmean().item():.1f} ml/kg/min")


Generating synthetic population...
                    mean        std         min         max
age            43.424400  15.047577   18.000000   69.000000
weight_kg      75.340130  14.707949   45.000000  126.810696
height_cm     171.727223   9.998051  145.000000  210.000000
resting_hr     64.691657  10.678714   35.000000  100.000000
exercise_hr    79.440437  68.623634    0.000000  191.581493
power_watts    12.760121  19.257410    0.000000   85.766866
duration_min   26.541757  27.132348    0.000000  114.134495
race_speed      1.172636   1.934153    0.000000    7.899769
vo2max         40.847161  13.399986   18.000000   88.000000
tau            52.570948   7.421060   25.616196   75.260285
C               0.329055   0.083327    0.064822    0.646266
P_w             0.823293   0.106927    0.460975    1.250675
L0              1.198625   0.300459    0.500000    2.360431
eta             0.279339   0.040405    0.150000    0.426695
alpha           0.598524   0.114700    0.400016    0.799888
beta 

In [ ]:
"""
MetabolicPINN — Training Pipeline
====================================
Run:
    python train.py                     # train on synthetic data
    python train.py --data real_data.csv --epochs 300
"""

import argparse
import os
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from pinn_model import MetabolicPINN, OUTPUT_NAMES, INPUT_FEATURES
from physics_loss import PhysicsLoss
from data_utils import (
    generate_synthetic_population, make_dataloaders,
    normalise, denormalise_tensor, TARGET_COLS
)

# ─────────────────────────────────────────────────────────────────────────────
#  Configuration defaults
# ─────────────────────────────────────────────────────────────────────────────
DEFAULT_CONFIG = {
    "epochs":          200,
    "batch_size":      128,
    "lr":              3e-4,
    "weight_decay":    1e-4,
    "hidden_dim":      256,
    "n_blocks":        6,
    "dropout":         0.1,
    "n_synthetic":     20_000,
    "lambda_data":     1.0,
    "lambda_ode1":     0.5,
    "lambda_ode2":     0.5,
    "lambda_power":    0.3,
    "lambda_emp":      0.8,
    "lambda_phys":     0.2,
    "checkpoint_dir":  "checkpoints",
    "patience":        20,          # early stopping patience
}


# ─────────────────────────────────────────────────────────────────────────────
#  Utility functions
# ─────────────────────────────────────────────────────────────────────────────

def get_device() -> torch.device:
    if torch.cuda.is_available():
        print("  CUDA GPU detected — using GPU.")
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        print("  Apple MPS detected — using MPS.")
        return torch.device("mps")
    else:
        print("  No GPU found — training on CPU (may be slow for large datasets).")
        return torch.device("cpu")


def evaluate(model, loader, loss_fn, device) -> tuple[float, dict]:
    model.eval()
    total_loss = 0.0
    comps_accum = {}
    n = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            params = model(x)
            loss, comps = loss_fn(params, x, y, lambda xn: denormalise_tensor(xn))
            total_loss += loss.item() * len(x)
            for k, v in comps.items():
                comps_accum[k] = comps_accum.get(k, 0.0) + v * len(x)
            n += len(x)

    avg_loss = total_loss / n
    avg_comps = {k: v / n for k, v in comps_accum.items()}
    return avg_loss, avg_comps


def compute_metrics(model, loader, device) -> dict:
    """Compute MAE on VO2max and threshold percentages."""
    model.eval()
    all_pred = []
    all_true = []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            params = model(x)
            all_pred.append(params.cpu())
            all_true.append(y.cpu())

    pred = torch.cat(all_pred, dim=0)   # (N, 9)
    true = torch.cat(all_true, dim=0)   # (N, 9) — may have NaNs

    metrics = {}
    for i, name in enumerate(TARGET_COLS):
        mask = ~torch.isnan(true[:, i])
        if mask.sum() > 0:
            mae = (pred[mask, i] - true[mask, i]).abs().mean().item()
            metrics[f"MAE_{name}"] = mae

    return metrics


def save_checkpoint(state: dict, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, model: MetabolicPINN, optimizer=None) -> int:
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    epoch = ckpt.get("epoch", 0)
    print(f"  Loaded checkpoint from epoch {epoch}")
    return epoch


# ─────────────────────────────────────────────────────────────────────────────
#  Main training function
# ─────────────────────────────────────────────────────────────────────────────

def train(config: dict, data_path: Optional[str] = None):
    device = get_device()
    print(f"\n{'='*60}")
    print("  MetabolicPINN — Training")
    print(f"{'='*60}\n")

    # ── Data ────────────────────────────────────────────────────────────────
    if data_path and os.path.exists(data_path):
        import pandas as pd
        print(f"  Loading real data from {data_path}")
        df = pd.read_csv(data_path)
        # Merge with synthetic to augment small real datasets
        df_synth = generate_synthetic_population(config["n_synthetic"] // 4)
        # Mark synthetic vs real (model doesn't need this, but useful for analysis)
        df["source"] = "real"
        df_synth["source"] = "synthetic"
        import pandas as pd2; df = pd2.concat([df, df_synth], ignore_index=True)
        print(f"  Real: {(df['source']=='real').sum():,}  +  Synthetic: {(df['source']=='synthetic').sum():,}")
    else:
        print(f"  No real data found — training on {config['n_synthetic']:,} synthetic samples.")
        print("  (Run with --data your_data.csv to add real measurements)")
        df = generate_synthetic_population(config["n_synthetic"])

    train_dl, val_dl, test_dl = make_dataloaders(df, batch_size=config["batch_size"])

    # ── Model ────────────────────────────────────────────────────────────────
    model = MetabolicPINN(
        hidden_dim=config["hidden_dim"],
        n_blocks=config["n_blocks"],
        dropout=config["dropout"],
    ).to(device)
    print(f"\n  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # ── Loss ─────────────────────────────────────────────────────────────────
    loss_fn = PhysicsLoss(
        lambda_data  = config["lambda_data"],
        lambda_ode1  = config["lambda_ode1"],
        lambda_ode2  = config["lambda_ode2"],
        lambda_power = config["lambda_power"],
        lambda_emp   = config["lambda_emp"],
        lambda_phys  = config["lambda_phys"],
    ).to(device)

    # ── Optimiser ────────────────────────────────────────────────────────────
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=1e-6)

    # ── Training loop ────────────────────────────────────────────────────────
    history = {"train_loss": [], "val_loss": [], "metrics": []}
    best_val_loss = float("inf")
    patience_counter = 0
    best_epoch = 0

    print(f"\n  Training for up to {config['epochs']} epochs  (patience={config['patience']})\n")
    print(f"  {'Epoch':>6}  {'Train':>10}  {'Val':>10}  {'ODE1':>8}  {'ODE2':>8}  {'EmpVO2':>8}  {'LR':>10}")
    print(f"  {'-'*70}")

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        train_loss = 0.0
        t0 = time.time()

        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            params = model(x)
            loss, comps = loss_fn(params, x, y, lambda xn: denormalise_tensor(xn.to(device)))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(x)

        train_loss /= len(train_dl.dataset)
        val_loss, val_comps = evaluate(model, val_dl, loss_fn, device)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        lr = scheduler.get_last_lr()[0]

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"  {epoch:>6}  {train_loss:>10.5f}  {val_loss:>10.5f}  "
                f"{val_comps.get('ode1',0):>8.5f}  {val_comps.get('ode2',0):>8.5f}  "
                f"{val_comps.get('emp_vo2',0):>8.5f}  {lr:>10.2e}"
            )

        # Early stopping & checkpointing
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            save_checkpoint(
                {"model_state": model.state_dict(),
                 "optimizer_state": optimizer.state_dict(),
                 "epoch": epoch, "config": config,
                 "val_loss": val_loss},
                f"{config['checkpoint_dir']}/best_model.pt"
            )
        else:
            patience_counter += 1
            if patience_counter >= config["patience"]:
                print(f"\n  Early stopping at epoch {epoch} (best: {best_epoch})")
                break

    # ── Final evaluation ─────────────────────────────────────────────────────
    print(f"\n  Loading best model (epoch {best_epoch})...")
    load_checkpoint(f"{config['checkpoint_dir']}/best_model.pt", model)
    metrics = compute_metrics(model, test_dl, device)

    print(f"\n{'='*60}")
    print("  Test Set Metrics (MAE):")
    for k, v in metrics.items():
        unit = "ml/kg/min" if "vo2" in k.lower() else ""
        print(f"    {k:25s}: {v:.4f} {unit}")
    print(f"{'='*60}\n")

    # Save history & config
    with open(f"{config['checkpoint_dir']}/training_history.json", "w") as f:
        json.dump({"history": history, "metrics": metrics, "config": config}, f, indent=2)

    print(f"  Training complete. Checkpoint saved to: {config['checkpoint_dir']}/best_model.pt")
    return model, history, metrics


# ─────────────────────────────────────────────────────────────────────────────
#  CLI Entry Point
# ─────────────────────────────────────────────────────────────────────────────

from typing import Optional

def parse_args():
    p = argparse.ArgumentParser(description="Train MetabolicPINN")
    p.add_argument("--data",       type=str,   default=None,       help="Path to real CSV data")
    p.add_argument("--epochs",     type=int,   default=200)
    p.add_argument("--batch_size", type=int,   default=128)
    p.add_argument("--lr",         type=float, default=3e-4)
    p.add_argument("--hidden_dim", type=int,   default=256)
    p.add_argument("--n_blocks",   type=int,   default=6)
    p.add_argument("--synthetic",  type=int,   default=20_000,     help="Synthetic samples")
    p.add_argument("--out_dir",    type=str,   default="checkpoints")

    # Use parse_known_args and only return the 'args' part (the first item in the tuple)
    args, _ = p.parse_known_args()
    return args


if __name__ == "__main__":
    # FIX: Call the function you just defined to get the args
    args = parse_args()

    config = DEFAULT_CONFIG.copy()
    config.update({
        "epochs":         args.epochs,
        "batch_size":     args.batch_size,
        "lr":             args.lr,
        "hidden_dim":     args.hidden_dim,
        "n_blocks":       args.n_blocks,
        "n_synthetic":    args.synthetic,
        "checkpoint_dir": args.out_dir,
    })
    train(config, data_path=args.data)

  CUDA GPU detected — using GPU.

  MetabolicPINN — Training

  No real data found — training on 20,000 synthetic samples.
  (Run with --data your_data.csv to add real measurements)
Dataset splits — Train: 16,000  Val: 3,000  Test: 1,000

  Model parameters: 797,193

  Training for up to 200 epochs  (patience=20)

   Epoch       Train         Val      ODE1      ODE2    EmpVO2          LR
  ----------------------------------------------------------------------
       1   116.87463   113.26328   0.00078   0.00001  80.45932    3.00e-04
      10    69.46142    86.65418   0.00084   0.00000  60.65784    2.98e-04


KeyboardInterrupt: 

In [ ]:
"""
MetabolicPINN — Gradio User Interface
======================================
Run:  python app.py
Opens a browser at http://localhost:7860
"""

import os
import numpy as np
import torch
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path

from pinn_model import MetabolicPINN, INPUT_FEATURES
from data_utils import normalise_single, denormalise_tensor
from pinn_model import vo2_trajectory, lactate_trajectory

# ─────────────────────────────────────────────────────────────────────────────
#  Load model
# ─────────────────────────────────────────────────────────────────────────────

CHECKPOINT = "checkpoints/best_model.pt"
_model = None

def get_model() -> MetabolicPINN:
    global _model
    if _model is not None:
        return _model

    _model = MetabolicPINN()

    if Path(CHECKPOINT).exists():
        ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
        _model.load_state_dict(ckpt["model_state"])
        print(f"  Loaded trained model from {CHECKPOINT}")
    else:
        print("  WARNING: No checkpoint found — running with random weights.")
        print("  Run `python train.py` first to train the model.")

    _model.eval()
    return _model


# ─────────────────────────────────────────────────────────────────────────────
#  Derived quantities helpers
# ─────────────────────────────────────────────────────────────────────────────

def hr_at_intensity(intensity_pct: float, hr_rest: float, hr_max: float) -> float:
    """Karvonen formula: HR = HRrest + pct * (HRmax - HRrest)"""
    return hr_rest + (intensity_pct / 100.0) * (hr_max - hr_rest)


def pace_from_vo2(vo2_target: float) -> str:
    """Approximate running pace (min/km) from VO2 using ACSM running equation."""
    # VO2 (ml/kg/min) = 3.5 + 0.2 * speed_m_min
    speed_m_min = max(1.0, (vo2_target - 3.5) / 0.2)
    speed_km_h  = speed_m_min * 60 / 1000
    min_per_km  = 60.0 / speed_km_h
    mins        = int(min_per_km)
    secs        = int((min_per_km - mins) * 60)
    return f"{mins}:{secs:02d} /km"


# ─────────────────────────────────────────────────────────────────────────────
#  Core prediction function
# ─────────────────────────────────────────────────────────────────────────────

def predict(
    age, weight_kg, height_cm, resting_hr,
    has_workout, exercise_hr, power_watts, duration_min,
    has_race, race_distance_km, race_time_min,
):
    # ── Input validation ────────────────────────────────────────────────────
    errors = []
    if age < 10 or age > 100:    errors.append("Age must be between 10 and 100.")
    if weight_kg < 30:           errors.append("Weight seems too low.")
    if height_cm < 100:          errors.append("Height seems too low.")
    if resting_hr < 30:          errors.append("Resting HR seems too low.")
    if errors:
        return "\n".join(errors), None, None

    # ── Race speed ─────────────────────────────────────────────────────────
    race_speed = 0.0
    if has_workout and race_distance_km > 0 and race_time_min > 0:
        race_speed = (race_distance_km * 1000) / (race_time_min * 60)  # m/s

    # ── Build feature dict ─────────────────────────────────────────────────
    features = {
        "age":          age,
        "weight_kg":    weight_kg,
        "height_cm":    height_cm,
        "resting_hr":   resting_hr,
        "exercise_hr":  exercise_hr if has_workout else 0.0,
        "power_watts":  power_watts if has_workout else 0.0,
        "duration_min": duration_min if has_workout else 0.0,
        "race_speed":   race_speed,
    }

    x = normalise_single(features)  # (1, 8)
    model = get_model()

    with torch.no_grad():
        params = model(x)

    # Unpack predictions
    vo2max   = params[0, 0].item()
    tau      = params[0, 1].item()
    C        = params[0, 2].item()
    P_w      = params[0, 3].item()
    L0       = params[0, 4].item()
    eta      = params[0, 5].item()
    alpha    = params[0, 6].item()
    beta     = params[0, 7].item()
    gamma    = params[0, 8].item()

    # ── Derive thresholds ──────────────────────────────────────────────────
    # Steady-state lactate = P_w / C
    L_ss = P_w / max(C, 1e-6)

    # AeT: L_ss = 2 mmol/L → scale factor
    aet_scale = min(max(2.0 / max(L_ss, 0.1), 0.50), 0.75)
    ant_scale = min(max(4.0 / max(L_ss, 0.1), 0.70), 0.92)

    vo2_aet = vo2max * aet_scale
    vo2_ant = vo2max * ant_scale
    aet_pct = (vo2_aet / max(vo2max, 1.0)) * 100
    ant_pct = (vo2_ant / max(vo2max, 1.0)) * 100

    # HR at thresholds
    hr_max = max(220 - age, 120)
    hr_aet = hr_at_intensity(aet_pct, resting_hr, hr_max)
    hr_ant = hr_at_intensity(ant_pct, resting_hr, hr_max)

    # Paces
    pace_aet = pace_from_vo2(vo2_aet)
    pace_ant = pace_from_vo2(vo2_ant)

    # FTP estimate (power at AnT ≈ 95% of functional threshold)
    # Using W = eta * (alpha * VO2 + beta * dL/dt + gamma)
    dL_aet = P_w * aet_scale - C * 2.0
    dL_ant = P_w * ant_scale - C * 4.0
    w_ant  = eta * (alpha * (vo2_ant * weight_kg / 1000 * 20.9) + beta * max(dL_ant, 0) + gamma)
    w_aet  = eta * (alpha * (vo2_aet * weight_kg / 1000 * 20.9) + beta * max(dL_aet, 0) + gamma)

    # ── Summary text ──────────────────────────────────────────────────────
    summary = f"""
╔══════════════════════════════════════════════════════════════╗
║             METABOLIC THRESHOLD PREDICTION REPORT            ║
╠══════════════════════════════════════════════════════════════╣
║  AEROBIC CAPACITY                                            ║
║  ─────────────────────────────────────────────────────────   ║
║  VO₂max                 : {vo2max:>6.1f} ml/kg/min           ║
║  VO₂ time constant (τ)  : {tau:>6.1f} s   (kinetic response) ║
║                                                              ║
║  AEROBIC THRESHOLD (AeT)  ≈ {aet_pct:.0f}% VO₂max            ║
║  ─────────────────────────────────────────────────────────   ║
║  VO₂ at AeT             : {vo2_aet:>6.1f} ml/kg/min          ║
║  Heart Rate at AeT      : {hr_aet:>6.0f} bpm                 ║
║  Running pace at AeT    : {pace_aet:>10s}                    ║
║  Lactate ~2 mmol/L      ← first threshold                    ║
║                                                              ║
║  ANAEROBIC THRESHOLD (AnT)  ≈ {ant_pct:.0f}% VO₂max          ║
║  ─────────────────────────────────────────────────────────   ║
║  VO₂ at AnT             : {vo2_ant:>6.1f} ml/kg/min          ║
║  Heart Rate at AnT      : {hr_ant:>6.0f} bpm                 ║
║  Running pace at AnT    : {pace_ant:>10s}                    ║
║  Lactate ~4 mmol/L      ← MLSS / OBLA                        ║
║                                                              ║
║  LACTATE KINETICS                                            ║
║  ─────────────────────────────────────────────────────────   ║
║  Clearance rate (C)     : {C:>6.3f} min⁻¹                    ║
║  Production rate (P_w)  : {P_w:>6.3f} mmol/L/min             ║
║  Resting lactate (L₀)   : {L0:>6.2f} mmol/L                  ║
║                                                              ║
║  METABOLIC EFFICIENCY                                        ║
║  ─────────────────────────────────────────────────────────   ║
║  Efficiency (η)         : {eta*100:>5.1f}%                   ║
╚══════════════════════════════════════════════════════════════╝
"""
    # Return values for plots
    plot_data = {
        "vo2max": vo2max, "tau": tau, "C": C, "P_w": P_w, "L0": L0,
        "eta": eta, "alpha": alpha, "beta": beta, "gamma": gamma,
        "vo2_aet": vo2_aet, "vo2_ant": vo2_ant,
        "aet_pct": aet_pct, "ant_pct": ant_pct,
        "hr_aet": hr_aet, "hr_ant": hr_ant,
        "hr_max": hr_max, "resting_hr": resting_hr,
        "weight_kg": weight_kg,
    }

    fig = make_plots(plot_data)
    return summary, fig, make_training_zone_table(vo2max, vo2_aet, vo2_ant, resting_hr, hr_max)


# ─────────────────────────────────────────────────────────────────────────────
#  Visualisations
# ─────────────────────────────────────────────────────────────────────────────

DARK_BG   = "#0d1117"
CARD_BG   = "#161b22"
ACCENT1   = "#58a6ff"   # blue
ACCENT2   = "#f78166"   # red/orange
ACCENT3   = "#3fb950"   # green
ACCENT4   = "#d2a8ff"   # purple
GRID_COL  = "#30363d"
TEXT_COL  = "#e6edf3"

plt.rcParams.update({
    "figure.facecolor":  DARK_BG,
    "axes.facecolor":    CARD_BG,
    "axes.edgecolor":    GRID_COL,
    "axes.labelcolor":   TEXT_COL,
    "xtick.color":       TEXT_COL,
    "ytick.color":       TEXT_COL,
    "grid.color":        GRID_COL,
    "text.color":        TEXT_COL,
    "font.family":       "DejaVu Sans",
    "axes.titlepad":     12,
})


def make_plots(d: dict) -> plt.Figure:
    fig = plt.figure(figsize=(16, 10), facecolor=DARK_BG)
    gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35,
                   left=0.07, right=0.97, top=0.92, bottom=0.08)

    _plot_vo2_kinetics(fig.add_subplot(gs[0, 0]), d)
    _plot_lactate_curve(fig.add_subplot(gs[0, 1]), d)
    _plot_zone_bar(fig.add_subplot(gs[0, 2]), d)
    _plot_threshold_intensity(fig.add_subplot(gs[1, 0:2]), d)
    _plot_metabolic_gauge(fig.add_subplot(gs[1, 2]), d)

    fig.suptitle("MetabolicPINN — Physiological Analysis", color=TEXT_COL,
                 fontsize=15, fontweight="bold", y=0.97)
    return fig


def _plot_vo2_kinetics(ax, d):
    t = torch.linspace(0, 360, 200)
    vo2max   = torch.tensor([d["vo2max"]])
    vo2_rest = torch.tensor([3.5])
    tau      = torch.tensor([d["tau"]])
    vo2_t    = vo2_trajectory(t, vo2max, vo2_rest, tau)[0].numpy()

    ax.plot(t.numpy(), vo2_t, color=ACCENT1, lw=2.5, label="VO₂(t)")
    ax.axhline(d["vo2max"],   color=ACCENT2, lw=1.2, ls="--", alpha=0.8, label=f"VO₂max = {d['vo2max']:.1f}")
    ax.axhline(d["vo2_aet"],  color=ACCENT3, lw=1.0, ls=":",  alpha=0.8, label=f"AeT = {d['vo2_aet']:.1f}")
    ax.axhline(d["vo2_ant"],  color=ACCENT4, lw=1.0, ls=":",  alpha=0.8, label=f"AnT = {d['vo2_ant']:.1f}")
    ax.axvline(d["tau"],      color=ACCENT1, lw=0.8, ls=":",  alpha=0.5)
    ax.text(d["tau"] + 5, 4.5, f"τ = {d['tau']:.0f}s", color=ACCENT1, fontsize=8)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("VO₂ (ml/kg/min)")
    ax.set_title("Eq 1 — VO₂ Kinetics", fontweight="bold")
    ax.legend(fontsize=7.5, loc="lower right")
    ax.grid(True, alpha=0.3)


def _plot_lactate_curve(ax, d):
    t = torch.linspace(0, 60, 200)
    P_w = torch.tensor([d["P_w"]])
    C   = torch.tensor([d["C"]])
    L0  = torch.tensor([d["L0"]])
    L_t = lactate_trajectory(t, P_w, C, L0)[0].numpy()

    ax.plot(t.numpy(), L_t, color=ACCENT2, lw=2.5, label="Lactate(t)")
    ax.axhline(2.0, color=ACCENT3, lw=1.2, ls="--", alpha=0.8, label="AeT ≈ 2 mmol/L")
    ax.axhline(4.0, color=ACCENT4, lw=1.2, ls="--", alpha=0.8, label="AnT ≈ 4 mmol/L")
    L_ss = d["P_w"] / max(d["C"], 1e-6)
    ax.axhline(L_ss, color=ACCENT2, lw=0.8, ls=":",  alpha=0.5, label=f"L_ss = {L_ss:.2f}")
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Blood Lactate (mmol/L)")
    ax.set_title("Eq 2 — Lactate Dynamics", fontweight="bold")
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(L_t.max() * 1.2, 5.0))


def _plot_zone_bar(ax, d):
    categories = ["Zone 1\n(Recovery)", "Zone 2\n(Aerobic)", "Zone 3\n(Tempo)",
                  "Zone 4\n(Threshold)", "Zone 5\n(VO₂max)"]
    colors = ["#2ea043", "#58a6ff", "#e3b341", "#f78166", "#db6d28"]

    aet_pct = d["aet_pct"]
    ant_pct = d["ant_pct"]
    boundaries = [0, aet_pct * 0.75, aet_pct, ant_pct, ant_pct * 1.04, 100]
    widths     = [boundaries[i+1] - boundaries[i] for i in range(5)]
    lefts      = boundaries[:-1]

    bars = ax.barh([0]*5, widths, left=lefts, color=colors, height=0.5, edgecolor=DARK_BG, linewidth=1.5)
    ax.axvline(aet_pct, color=ACCENT3, lw=2, ls="--")
    ax.axvline(ant_pct, color=ACCENT4, lw=2, ls="--")
    ax.text(aet_pct + 1, 0.3, f"AeT\n{aet_pct:.0f}%", color=ACCENT3, fontsize=8, va="center")
    ax.text(ant_pct + 1, 0.3, f"AnT\n{ant_pct:.0f}%", color=ACCENT4, fontsize=8, va="center")

    ax.set_xlim(0, 110)
    ax.set_yticks([])
    ax.set_xlabel("% VO₂max")
    ax.set_title("Training Zone Distribution", fontweight="bold")

    patches = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, categories)]
    ax.legend(handles=patches, fontsize=7, loc="lower right", ncol=2)
    ax.grid(True, axis="x", alpha=0.3)


def _plot_threshold_intensity(ax, d):
    hr_rest = d["resting_hr"]
    hr_max  = d["hr_max"]
    pcts    = np.linspace(0, 100, 100)
    hrs     = hr_rest + pcts / 100 * (hr_max - hr_rest)
    vo2s    = d["vo2max"] * pcts / 100

    ax.plot(hrs, vo2s, color=ACCENT1, lw=2.5, label="VO₂ vs HR")

    # AeT marker
    ax.axvline(d["hr_aet"], color=ACCENT3, lw=1.5, ls="--", alpha=0.9)
    ax.axhline(d["vo2_aet"], color=ACCENT3, lw=1.5, ls="--", alpha=0.9)
    ax.scatter(d["hr_aet"], d["vo2_aet"], color=ACCENT3, s=80, zorder=5,
               label=f"AeT — {d['hr_aet']:.0f} bpm / {d['vo2_aet']:.1f} ml/kg/min")

    # AnT marker
    ax.axvline(d["hr_ant"], color=ACCENT4, lw=1.5, ls="--", alpha=0.9)
    ax.axhline(d["vo2_ant"], color=ACCENT4, lw=1.5, ls="--", alpha=0.9)
    ax.scatter(d["hr_ant"], d["vo2_ant"], color=ACCENT4, s=80, zorder=5,
               label=f"AnT — {d['hr_ant']:.0f} bpm / {d['vo2_ant']:.1f} ml/kg/min")

    ax.set_xlabel("Heart Rate (bpm)")
    ax.set_ylabel("VO₂ (ml/kg/min)")
    ax.set_title("Eq 3 — Threshold Intensity Map (HR vs VO₂)", fontweight="bold")
    ax.legend(fontsize=8.5)
    ax.grid(True, alpha=0.3)


def _plot_metabolic_gauge(ax, d):
    # Radial gauge for VO2max fitness category
    vo2 = d["vo2max"]

    # Fitness categories (ml/kg/min, approximate mixed sex)
    thresholds = [20, 30, 40, 50, 60, 75, 90]
    labels     = ["Poor", "Fair", "Average", "Good", "Excellent", "Elite", ""]
    colors_g   = ["#f85149", "#f78166", "#e3b341", "#58a6ff", "#3fb950", "#d2a8ff"]

    theta_start, theta_end = np.pi, 0  # half-circle
    n_seg = len(colors_g)
    thetas = np.linspace(theta_start, theta_end, n_seg + 1)

    for i in range(n_seg):
        ax.bar(x=(thetas[i] + thetas[i+1]) / 2, height=0.3,
               width=thetas[i+1] - thetas[i],
               bottom=0.65, color=colors_g[i], alpha=0.7,
               edgecolor=DARK_BG, linewidth=1.5)

    # Needle
    pct_along = (vo2 - thresholds[0]) / (thresholds[-2] - thresholds[0])
    pct_along = np.clip(pct_along, 0, 1)
    needle_theta = theta_start + pct_along * (theta_end - theta_start)
    ax.annotate("", xy=(np.cos(needle_theta) * 0.8, np.sin(needle_theta) * 0.8),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=TEXT_COL, lw=2))
    ax.plot(0, 0, "o", color=TEXT_COL, ms=8, zorder=10)

    # Category text
    cat_idx = np.searchsorted(thresholds[1:-1], vo2)
    cat_label = labels[min(cat_idx, len(labels)-2)]
    ax.text(0, 0.25, f"{vo2:.1f}", ha="center", va="center",
            fontsize=20, fontweight="bold", color=TEXT_COL)
    ax.text(0, -0.15, "ml/kg/min", ha="center", va="center", fontsize=9, color=TEXT_COL, alpha=0.8)
    ax.text(0, -0.40, cat_label, ha="center", va="center",
            fontsize=13, fontweight="bold", color=colors_g[min(cat_idx, len(colors_g)-1)])

    # Labels
    for i, lab in enumerate(labels[:-1]):
        theta = (thetas[i] + thetas[i+1]) / 2
        ax.text(np.cos(theta) * 1.05, np.sin(theta) * 1.05, lab,
                ha="center", va="center", fontsize=6.5, color=TEXT_COL, alpha=0.7)

    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-0.6, 1.1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("VO₂max Fitness Category", fontweight="bold")


def make_training_zone_table(vo2max, vo2_aet, vo2_ant, hr_rest, hr_max) -> str:
    zones = [
        ("Zone 1", "Recovery",   0.60 * vo2max, 0.65 * vo2max),
        ("Zone 2", "Aerobic",    vo2_aet * 0.75, vo2_aet),
        ("Zone 3", "Tempo",      vo2_aet,         vo2_ant * 0.92),
        ("Zone 4", "Threshold",  vo2_ant * 0.92,  vo2_ant),
        ("Zone 5", "VO₂max",     vo2_ant,          vo2max),
    ]

    lines = ["PERSONALISED TRAINING ZONES\n"]
    lines.append(f"{'Zone':<8}  {'Name':<12}  {'VO₂ range':>18}  {'HR range':>16}  {'Purpose'}")
    lines.append("─" * 82)

    purposes = [
        "Active recovery, fat metabolism",
        "Base fitness, primary aerobic zone",
        "Race pace, lactate clearance",
        "Threshold development",
        "Max power, anaerobic capacity",
    ]

    for (z, name, v_lo, v_hi), purpose in zip(zones, purposes):
        hr_lo = hr_at_intensity((v_lo / max(vo2max, 1)) * 100, hr_rest, hr_max)
        hr_hi = hr_at_intensity((v_hi / max(vo2max, 1)) * 100, hr_rest, hr_max)
        lines.append(
            f"{z:<8}  {name:<12}  {v_lo:6.1f}–{v_hi:5.1f} ml/kg/min  "
            f"{hr_lo:5.0f}–{hr_hi:4.0f} bpm  {purpose}"
        )

    return "\n".join(lines)


# ─────────────────────────────────────────────────────────────────────────────
#  Gradio Interface
# ─────────────────────────────────────────────────────────────────────────────

CUSTOM_CSS = """
body { background: #0d1117 !important; }
.gradio-container { max-width: 1100px; margin: auto; }
.gr-box { border-radius: 10px !important; border: 1px solid #30363d !important; }
h1, h2, h3 { color: #58a6ff !important; }
.gr-button-primary { background: #238636 !important; border: 1px solid #2ea043 !important; }
.gr-button-primary:hover { background: #2ea043 !important; }
#title-banner {
    background: linear-gradient(135deg, #0d1117 0%, #161b22 50%, #0d1117 100%);
    border: 1px solid #30363d;
    border-radius: 12px;
    padding: 20px;
    text-align: center;
    margin-bottom: 20px;
}
"""

DESCRIPTION = """
<div id="title-banner">
<h1 style="color:#58a6ff; font-size:2em; margin:0;">MetabolicPINN</h1>
<p style="color:#8b949e; font-size:1.05em; margin-top:8px;">
Physics-Informed Neural Network for Metabolic Threshold Prediction<br>
<em>Governing equations: VO₂ kinetics · Lactate dynamics · Metabolic power constraint</em>
</p>
</div>
"""

def toggle_workout(has_workout):
    return gr.update(visible=has_workout)

def toggle_race(has_race):
    return gr.update(visible=has_race)


def build_interface():
    with gr.Blocks(css=CUSTOM_CSS, title="MetabolicPINN") as demo:
        gr.HTML(DESCRIPTION)

        with gr.Row():
            # ── LEFT COLUMN: Inputs ─────────────────────────────────────────
            with gr.Column(scale=1):
                gr.Markdown("###  Personal Profile")
                with gr.Group():
                    age        = gr.Slider(15, 80, value=30, step=1,  label="Age (years)")
                    weight_kg  = gr.Slider(40, 150, value=75, step=0.5, label="Weight (kg)")
                    height_cm  = gr.Slider(140, 220, value=175, step=1, label="Height (cm)")
                    resting_hr = gr.Slider(35, 100, value=65, step=1,  label="Resting Heart Rate (bpm)")

                gr.Markdown("###  Exercise Data (optional)")
                has_workout = gr.Checkbox(label="I have workout data", value=False)
                with gr.Group(visible=False) as workout_group:
                    exercise_hr  = gr.Slider(60, 220, value=150, step=1,  label="Average Exercise HR (bpm)")
                    power_watts  = gr.Slider(0, 600,  value=200, step=5,  label="Average Power (watts)  — set 0 if running/unknown")
                    duration_min = gr.Slider(1, 300,  value=45,  step=1,  label="Exercise Duration (min)")

                gr.Markdown("###  Race / Time Trial (optional)")
                has_race = gr.Checkbox(label="I have race data", value=False)
                with gr.Group(visible=False) as race_group:
                    race_distance = gr.Dropdown(
                        choices=["1K (1 km)", "5K (5 km)", "10K (10 km)", "Half Marathon (21.1 km)", "Marathon (42.2 km)"],
                        value="5K (5 km)", label="Race Distance"
                    )
                    race_time_min = gr.Slider(5, 300, value=25, step=0.5, label="Race Time (minutes)")

                predict_btn = gr.Button(" Predict Metabolic Thresholds", variant="primary", size="lg")

            # ── RIGHT COLUMN: Outputs ───────────────────────────────────────
            with gr.Column(scale=2):
                gr.Markdown("###  Results")
                summary_text = gr.Textbox(
                    label="Metabolic Report",
                    lines=28, max_lines=35,
                    show_copy_button=True,
                )
                gr.Markdown("###  Physiological Plots")
                plot_output = gr.Plot(label="Analysis Charts")
                gr.Markdown("###  Training Zones")
                zones_text = gr.Textbox(
                    label="Personalised Training Zones",
                    lines=12, max_lines=15,
                    show_copy_button=True,
                )

        # ── Toggle visibility ──────────────────────────────────────────────
        has_workout.change(toggle_workout, has_workout, workout_group)
        has_race.change(toggle_race, has_race, race_group)

        # ── Wire up predict ────────────────────────────────────────────────
        DIST_MAP = {
            "1K (1 km)": 1.0, "5K (5 km)": 5.0, "10K (10 km)": 10.0,
            "Half Marathon (21.1 km)": 21.1, "Marathon (42.2 km)": 42.2,
        }

        def run_prediction(age, wt, ht, rhr, hw, ehr, pw, dm, hr, rd, rtm):
            dist_km = DIST_MAP.get(rd, 5.0)
            return predict(age, wt, ht, rhr, hw, ehr, pw, dm, hr, dist_km, rtm)

        predict_btn.click(
            run_prediction,
            inputs=[age, weight_kg, height_cm, resting_hr,
                    has_workout, exercise_hr, power_watts, duration_min,
                    has_race, race_distance, race_time_min],
            outputs=[summary_text, plot_output, zones_text],
        )

        gr.Markdown("""
---
> **Disclaimer:** Predictions are based on physics-informed modelling and empirical equations.
> For clinical or competitive use, validate with a professional CPET (cardiopulmonary exercise test).
> Model accuracy improves significantly after training on real lactate/VO₂max datasets.
        """)

    return demo


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--share", action="store_true", help="Create public Gradio link")
    parser.add_argument("--port",  type=int, default=7860)

    args, _ = parser.parse_known_args()

    demo = build_interface()

    demo.launch(server_port=args.port, share=True, show_error=True)


/tmp/ipykernel_216/1779484188.py:459: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="MetabolicPINN") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://37a5951353097fe784.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
